# Celestack Workflow

This is a development workflow for the `celestack` project.

Before release, it will be fully substituted by a CLI, TUI, or GUI.

## Project-level Settings:

In [ ]:
from pathlib import Path

from celestack.stack import FrameStack

PROJECT = "test"

## Initialize the Project Stack

In [ ]:
stack = FrameStack(PROJECT)
stack

## Load the Frames

In [ ]:
stack = FrameStack.from_state(PROJECT)
lf_paths = list(Path("/home/martin/Desktop/tenerife/lf").glob("*.tif"))
df_paths = list(Path("/home/martin/Desktop/tenerife/df").glob("*.tif"))

stack.load_frames(
    lf_paths=lf_paths,
    df_paths=df_paths,
)

stack = FrameStack.from_state(PROJECT)
print(f"{len(stack.light_frames)=}, {len(stack.dark_frames)=}")

## Apply the Dark Frames Correction

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.apply_dark_frames_correction()

stack = FrameStack.from_state(PROJECT)
stack.master_dark

## Create the Average Light Frame

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.create_average_light_frame()

stack = FrameStack.from_state(PROJECT)
stack.avg_light

## Create and Apply the Foreground Mask

Start with clustering the pixels of the average light frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)
if stack.avg_light is None:
    raise ValueError("No average light frame found.")

alf = stack.avg_light
alf.cluster_pixels(n_clusters=2)
fig = alf.plot_clusters()
fig.show()


Now, turn the clustered pixels into a `Mask` object and add it to the stack.

In [ ]:
alf.initialize_mask(foreground_cluster_labels=[0])

# Explicitly mask/unmask some areas of the image
alf.set_mask_in_box(False, y2=1760)  # top part of the image is sky
alf.set_mask_in_box(True, y1=2035)  # bottom part of the image is foreground

mask = alf.create_mask()

stack.add_mask(mask)

And finally, add the mask to the stack.

In [ ]:


# Show that the mask has been added to the stack
stack = FrameStack.from_state(PROJECT)
if stack.mask is None or stack.light_frames["P3290058"].mask is None:
    raise ValueError("No mask found in stack or light frame.")

stack.mask.plot().show()

## Sky Segmentation

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.segment_sky()

stack = FrameStack.from_state(PROJECT)
if not stack.segment_boxes:
    raise ValueError("No segment boxes found in stack.")

stack = FrameStack.from_state(PROJECT)
stack.plot().show()

## Detect Stars in Reference Frame

First, set the reference frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.set_reference_frame("P3290100")

Now, detect stars in the reference frame.

In [ ]:
stack = FrameStack.from_state(PROJECT)
stack.detect_stars_in_ref_frame(n=1000)

Let's plot the stars.

In [ ]:
stack = FrameStack.from_state(PROJECT)

fig = stack.plot()
fig.show()

stack.stars_table

## Propagate the stars across the stack

Let's test the algo for propagating a single star across the stack.

In [ ]:
import pandas as pd
import numpy as np

from tqdm import tqdm
import plotly.express as px

assert stack.stars_table is not None
assert stack.ref_frame is not None

def get_t_series(stack: FrameStack) -> pd.Series:
    """Get the time of all the frames in the stack.

    Get the time-like distance of the frame from the reference frame.
    If the frames have exif data with the timestamp, we'll use that and the `t` will
    be in seconds.
    Otherwise, we'll raise a NotImplementedError.

    Args:
        stack: The frame stack.
        frame_name: The name of the frame.

    Returns:
        The series of time of capture for each frame, relative to the reference frame.
        The t is in [seconds] and the t of the reference frame is 0.
        The output is a pandas Series with the frame names as index, times as values and
        the name "t".
    """
    try:
        timestamps: dict[str, str] = {
            frame_name: frame.exif_data["DateTimeOriginal"]
            for frame_name, frame in stack.light_frames.items()
        }
    except KeyError:
        raise NotImplementedError(
            "The frames do not have exif data with the timestamp. "
            "Please implement a way to get the time of capture for each frame."
        )
    
    times = pd.Series(timestamps)
    times = pd.to_datetime(times, format="%Y:%m:%d %H:%M:%S")

    # Convert to seconds relative to the reference frame
    times = (times - times[stack.ref_frame.name]).dt.total_seconds()  # type: ignore
    times.name = "t"

    return times


def predict_xy(
    star_coordinates: pd.DataFrame, frame_name: str, n_closest: int = 5,
    ) -> tuple[float, float]:
    """Predict the star position in a frame, based on already know positions in others.

    If only the position is known only for a single frame (the reference frame),
    then this position is used as the predicted position in the next frame.
    This is only possible, if the frame in question is the closest neighbor of the
    reference frame, otherwise the function returns None.

    If more than one position is known, then the N closest positions in star_coordinates
    are fitted with a linear fit and the position is extrapolated to the frame in
    question.

    Args:
        star_coordinates: The DataFrame with the star coordinates.
            The DataFrame must have the columns "t", "x" and "y" and the index must be
            the frame names.
            The x, y are pixel coordinates of the star in the respective frame.
            The t is the time of capture of the frame, relative to the reference frame
            (in seconds or any other arbitrary time unit).
        frame_name: The name of the frame to predict the position for.
        n_closest: Only the N closest stars to the frame in question are used for the
            fit to extrapolate the position of the star in the frame in question.

    Returns:
        The predicted x and y coordinates of the star in the frame with the name
        `frame_name`.
        The coordinates are in pixels in that frame.
        If the prediction is not possible, None is returned.
    """
    # Take care of the base case - in only a single frame has the star coordinates:
    if len(star_coordinates.x.dropna()) == 1:
        # What is the loc index of the frame with the known coordinates?
        known_name: str = star_coordinates.x.dropna().index[0]  # type: ignore
        i_known: int = star_coordinates.index.get_loc(known_name)  # type: ignore
        # Is the `frame_name` the closest neighbor of the known frame?
        i_frame: int = star_coordinates.index.get_loc(frame_name)  # type: ignore
        if abs(i_known - i_frame) <= 1:
            # The frame is the closest neighbor of the reference frame
            return star_coordinates.x[known_name], star_coordinates.y[known_name]
        else:
            # The frame is not the closest neighbor of the reference frame
            return np.nan, np.nan
    
    # If we have coordinates of the star for more than one frame, we'll fit a line
    # through the points and extrapolate the position to the frame in question.
    t_frame: float = star_coordinates.t[frame_name]
    coords = star_coordinates.dropna()

    if len(coords) == 2:
        # If we only have two frames, they define the line fully:
        (t0, x0, y0), (t1, x1, y1) = coords.values

        a_x = (x1 - x0) / (t1 - t0)
        b_x = x0 - a_x * t0

        a_y = (y1 - y0) / (t1 - t0)
        b_y = y0 - a_y * t0

        return a_x * t_frame + b_x, a_y * t_frame + b_y

    # We'll fit a line through the points...
    # We'll only use N closest frames to the frame in question
    if len(coords) > n_closest:
        coords = coords.loc[
            (coords["t"] - t_frame).abs().nsmallest(n_closest).index
        ]
    x = coords.x.to_numpy().astype(float)
    y = coords.y.to_numpy().astype(float)
    t = coords.t.to_numpy().astype(float)
    
    # Fit linear models
    a_x, b_x = np.polyfit(t, x, 1)
    a_y, b_y = np.polyfit(t, y, 1)
    
    return a_x * t_frame + b_x, a_y * t_frame + b_y


# Build the series of times of all the frames in the stack
ref_name = stack.ref_frame.name
t_series = get_t_series(stack)


# Propagate all the stars in the stack:
for star_id in tqdm(stack.stars_table.id.unique(), desc="Propagating stars"):

    # Initialize the star coordinates DataFrame, indexed by frame names:
    star_table = pd.DataFrame(
        index=list(stack.light_frames), 
        columns=["t", "x", "y", "flux", "threshold", "fwhm"],
    )
    star_table["t"] = t_series
    x0, y0 = stack.stars_table.loc[star_id, ["x", "y"]]  # type: ignore
    fwhm = stack.stars_table.at[star_id, "fwhm"]
    threshold = stack.stars_table.at[star_id, "threshold"]

    star_table.loc[ref_name, ["x", "y"]] = [x0, y0]

    # Sort the star coordinates by time
    star_table = star_table.sort_values(by="t", key=lambda x: x.abs())

    # Predict the star position for all the frames (in two passes):
    # TODO: Need to distinguish between stars in foreground/out-of-frame and failed det.
    for _ in range(2):
        for frame_name in star_table.index:
            if pd.isna(star_table.loc[frame_name, "x"]):
                # start with the linear extrapolation as a first guess:
                x, y = predict_xy(star_table[["t", "x", "y"]], frame_name)
                if np.isnan(x) or np.isnan(y):
                    continue
                # Find the star in the frame:
                frame = stack.light_frames[frame_name]
                star = frame.find_star(x, y, fwhm, threshold, 1.7, 1.5)
                # TODO: The roundness and the distance are hardcoded here - bad!
                if star is not None:
                    # If the star was found, we'll use its coordinates:
                    star_table.loc[
                        frame_name, ["x", "y", "flux", "threshold", "fwhm"]
                    ] = [
                        star["x"],
                        star["y"],
                        star["flux"],
                        star["threshold"],
                        star["fwhm"]
                    ]

    # Finish the star table so it conforms to the stack.stars_table format:
    star_table = star_table.drop(index=ref_name)  # already in the stack
    star_table.index.name = "frame"
    star_table = star_table.reset_index()
    star_table["id"] = star_id

    # Add the star table to the stack:
    stack.stars_table = pd.concat(
        [stack.stars_table, star_table], ignore_index=True
    )
    stack.dump_state()


## Development

The problematic stars:

* 334, 876: The brightest stars, not propagated at all
* 833: One of the stars in the low-right corner which goes loopy

In [ ]:
stack = FrameStack.from_state(PROJECT)

stack.plot()